In [3]:
import os
import numpy as np
import pandas as pd
# import seaborn as sn
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import matplotlib.pyplot as plt
# import nltk
# from nltk.tokenize import word_tokenize
# from nltk.stem import WordNetLemmatizer
import re
import gc
from sklearn.metrics import confusion_matrix, classification_report
from transformers import BertConfig, BertModel, AutoTokenizer, AutoConfig, AutoModel
import evaluate
from collections import Counter
# import sentencepiece

In [2]:
device = 'cuda:0'

In [24]:
# Load the pre-trained BERT model and tokenizer
config = BertConfig.from_pretrained("HooshvareLab/bert-base-parsbert-uncased")
tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-base-parsbert-uncased")
model = BertModel.from_pretrained("HooshvareLab/bert-base-parsbert-uncased")

Some weights of the model checkpoint at HooshvareLab/bert-base-parsbert-uncased were not used when initializing BertModel: ['cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.decoder.weight']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### Preprocessing for ParsBERT

In [4]:
def data_prep(question, context, is_impossible, start_index, stop_index, device):
    encoding = tokenizer.encode_plus(question, context, max_length = 512, truncation=True,
                                     return_tensors='pt', return_offsets_mapping=True)
    offsets = encoding['offset_mapping']
    token_start_index = 0
    token_stop_index = 0
    if is_impossible:
        return [encoding['input_ids'].to(device), encoding['token_type_ids'].to(device), encoding['attention_mask'].to(device), torch.tensor([0], device=device), torch.tensor([0], device=device)]
    for i, (start, end) in enumerate(offsets[0].tolist()):
        if start_index >= start and start_index < end:
            token_start_index = i
        if stop_index > start and stop_index <= end:
            token_stop_index = i
    return [encoding['input_ids'].to(device), encoding['token_type_ids'].to(device), encoding['attention_mask'].to(device), torch.tensor([token_start_index], device=device), torch.tensor([token_stop_index], device=device)]

In [5]:
train_data_raw_pd = pd.read_json('Dataset/Train.json')
train_data_raw_np = train_data_raw_pd.to_numpy()

train_data = []
for row in train_data_raw_np[:,1]:
    for single_context_all_qans in row['paragraphs']:
        context = single_context_all_qans['context']
        for qans in single_context_all_qans['qas']:
            if qans['is_impossible']:
                start_ans_token = 0
                stop_ans_token = 0
                prepared_1data = data_prep(question, context, True, start_ans_idx, stop_ans_idx, device)
                train_data.append(prepared_1data)
                continue
            question = qans['question']
            start_ans_idx = qans['answers'][0]['answer_start']
            stop_ans_idx = start_ans_idx + len(qans['answers'][0]['text'])
            prepared_1data = data_prep(question, context, False, start_ans_idx, stop_ans_idx, device)
            train_data.append(prepared_1data)
            # print(start_ans_idx, stop_ans_idx, question)
        # break
    # break

len(train_data)

63994

In [6]:
validation_data_raw_pd = pd.read_json('Dataset/Validation.json')
validation_data_raw_np = validation_data_raw_pd.to_numpy()

validation_data = []
for row in validation_data_raw_np[:,1]:
    for single_context_all_qans in row['paragraphs']:
        context = single_context_all_qans['context']
        for qans in single_context_all_qans['qas']:
            if qans['is_impossible']:
                start_ans_token = 0
                stop_ans_token = 0
                prepared_1data = data_prep(question, context, True, start_ans_idx, stop_ans_idx, device)
                validation_data.append(prepared_1data)
                continue
            question = qans['question']
            start_ans_idx = qans['answers'][0]['answer_start']
            stop_ans_idx = start_ans_idx + len(qans['answers'][0]['text'])
            prepared_1data = data_prep(question, context, False, start_ans_idx, stop_ans_idx, device)
            validation_data.append(prepared_1data)
            # print(start_ans_idx, stop_ans_idx, question)
        # break
    # break

len(validation_data)

7976

In [7]:
test_data_raw_pd = pd.read_json('Dataset/Test.json')
test_data_raw_np = test_data_raw_pd.to_numpy()

test_data = []
for row in test_data_raw_np[:,1]:
    for single_context_all_qans in row['paragraphs']:
        context = single_context_all_qans['context']
        for qans in single_context_all_qans['qas']:
            if qans['is_impossible']:
                start_ans_token = 0
                stop_ans_token = 0
                prepared_1data = data_prep(question, context, True, start_ans_idx, stop_ans_idx, device)
                test_data.append(prepared_1data)
                continue
            question = qans['question']
            start_ans_idx = qans['answers'][0]['answer_start']
            stop_ans_idx = start_ans_idx + len(qans['answers'][0]['text'])
            prepared_1data = data_prep(question, context, False, start_ans_idx, stop_ans_idx, device)
            test_data.append(prepared_1data)
            # print(start_ans_idx, stop_ans_idx, question)
        # break
    # break

len(test_data)

8002

In [87]:
def data_stats(data):
    len_data = len(data)
    cls_token = torch.tensor([3], device=device)
    data_q_tokens_num = torch.zeros(len_data, device=device)
    data_p_tokens_num = torch.zeros(len_data, device=device)
    data_a_tokens_num = torch.zeros(len_data, device=device)
    non_ans_num = 0
    for i in range(len_data):
        q_tokens_num = 0
        for token in data[i][0][0]:
            if token == cls_token:
                break
            q_tokens_num += 1
        q_tokens_num -= 1
        p_tokens_num = len(data[i][0][0]) - q_tokens_num - 3
        a_start = data[i][4]
        a_stop = data[i][3]
        if a_start == 0 and a_stop == 0:
            non_ans_num += 1
            a_tokens_num = 0
        else:  
            a_tokens_num = (a_start - a_stop).item() + 1
        data_q_tokens_num[i] = q_tokens_num
        data_p_tokens_num[i] = p_tokens_num
        data_a_tokens_num[i] = a_tokens_num
        # break
    mean_p_tokens_num = data_p_tokens_num.mean()
    mean_q_tokens_num = data_q_tokens_num.mean()
    mean_a_tokens_num = data_a_tokens_num.sum() / (len_data - non_ans_num)
    max_p_tokens = data_p_tokens_num.max()
    max_q_tokens = data_q_tokens_num.max()
    max_a_tokens = data_a_tokens_num.max()
    print(f'Total Questions = {len_data}\nUnanswerable Questions = {non_ans_num}\nMean of paragraph tokens = {int(mean_p_tokens_num)}\nMean of question tokens = {int(mean_q_tokens_num)}\nMean of answer tokens = {int(mean_a_tokens_num)}')
    print(f'Max of paragraph tokens = {int(max_p_tokens)}\nMax of question tokens = {int(max_q_tokens)}\nMax of answer tokens = {int(max_a_tokens)}')

    return [len_data, non_ans_num, mean_p_tokens_num, mean_q_tokens_num, mean_a_tokens_num, max_p_tokens, max_q_tokens, max_a_tokens]

In [88]:
train_stats = data_stats(train_data)

Total Questions = 63994
Unanswerable Questions = 15721
Mean of paragraph tokens = 167
Mean of question tokens = 14
Mean of answer tokens = 6
Max of paragraph tokens = 452
Max of question tokens = 95
Max of answer tokens = 268


In [89]:
val_stats = data_stats(validation_data)

Total Questions = 7976
Unanswerable Questions = 1981
Mean of paragraph tokens = 161
Mean of question tokens = 15
Mean of answer tokens = 7
Max of paragraph tokens = 333
Max of question tokens = 60
Max of answer tokens = 201


In [90]:
test_stats = data_stats(test_data)

Total Questions = 8002
Unanswerable Questions = 1914
Mean of paragraph tokens = 165
Mean of question tokens = 15
Mean of answer tokens = 7
Max of paragraph tokens = 394
Max of question tokens = 69
Max of answer tokens = 161


In [96]:
print('Total Data')
total_q = train_stats[0]+val_stats[0]+test_stats[0]
print('Total Questions =', total_q)
unans_q = train_stats[1]+val_stats[1]+test_stats[1]
print('Unanswerable Questions =', unans_q)
mean_p = (train_stats[0]*train_stats[2]+val_stats[0]*val_stats[2]+test_stats[0]*test_stats[2]) / total_q
print('Mean of paragraph tokens =', int(mean_p))
mean_q = (train_stats[0]*train_stats[3]+val_stats[0]*val_stats[3]+test_stats[0]*test_stats[3]) / total_q
print('Mean of question tokens =', int(mean_q))
mean_a = ((train_stats[0]-train_stats[1])*train_stats[4]+(val_stats[0]-val_stats[1])*val_stats[4]+(test_stats[0]-test_stats[1])*test_stats[4]) / (total_q-unans_q)
print('Mean of question tokens =', int(mean_a))

max_p = max(train_stats[5],val_stats[5],test_stats[5])
print('Max of paragraph tokens =', int(max_p))
max_q = max(train_stats[6],val_stats[6],test_stats[6])
print('Max of question tokens =', int(max_q))
max_a = max(train_stats[7],val_stats[7],test_stats[7])
print('Max of answer tokens =', int(max_a))

Total Data
Total Questions = 79972
Unanswerable Questions = 19616
Mean of paragraph tokens = 166
Mean of question tokens = 14
Mean of question tokens = 6
Max of paragraph tokens = 452
Max of question tokens = 95
Max of answer tokens = 268


In [8]:
indices = torch.randperm(len(train_data))
train_data_sh = [train_data[i] for i in indices]

# Training

## ParsBERT

In [9]:
# model

In [10]:
gc.collect()

0

In [25]:
class PosModel(nn.Module):
    def __init__(self):
        super(PosModel, self).__init__()
        
        self.base_model = model
        self.linear = torch.nn.Sequential(
                      torch.nn.Linear(768, 2),
                        ) 
        
    def forward(self, input_ids, token_type_ids, attention_mask):
        outputs = self.base_model(input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        outputs = self.linear(outputs[0])
        
        return outputs

In [26]:
gc.collect()
torch.cuda.empty_cache()

In [27]:
gc.collect()
torch.cuda.empty_cache()

lr = 0.00001
batch_size = 8
torch.manual_seed(46)
# Initialize the model, the optimizer and criterion
parsbert_model = PosModel()
parsbert_model = parsbert_model.to(device)
optimizer = optim.AdamW(parsbert_model.parameters(), lr=lr, eps=1e-08)
criterion = nn.CrossEntropyLoss()

train_loss = np.zeros((0))
test_loss = np.zeros((0))
total_epoch_counter = 0
total_epoch = 0
loader_size = len(train_data_sh)//batch_size

In [28]:
num_epochs = 1
total_epoch += num_epochs

for epoch in range(num_epochs):
    
    parsbert_model.train()
    total_loss = 0
    loss_token_start = torch.tensor([0.0],device=device)
    loss_token_stop = torch.tensor([0.0],device=device)
    optimizer.zero_grad()
    for i in range(loader_size):
        torch.cuda.empty_cache()
        for j in range(batch_size):
            input_data = train_data_sh[i*batch_size+j][0]
            token_type_ids = train_data_sh[i*batch_size+j][1]
            attention_mask = train_data_sh[i*batch_size+j][2]
            token_start_true = train_data_sh[i*batch_size+j][3]
            token_stop_true = train_data_sh[i*batch_size+j][4]
            outputs = parsbert_model(input_data, token_type_ids, attention_mask)
            loss_token_start += criterion(outputs[:,:,0], token_start_true)
            loss_token_stop += criterion(outputs[:,:,1], token_stop_true)
        loss_token_start /= batch_size
        loss_token_stop /= batch_size
        loss = loss_token_start + loss_token_stop
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_token_start = torch.tensor([0.0],device=device)
        loss_token_stop = torch.tensor([0.0],device=device)
        total_loss += loss
        if (i+1)*batch_size > 32000:
            break
        # total_loss /= loader_size
    
        if (i+1) % 40 == 0:
            total_loss /= 40
            print('Loader Num: [{}/{}], Train mean loss: {:.7f}, lr: {:.6f}'
                   .format(i+1, loader_size, total_loss.detach().item(), lr))
            total_loss = 0
    total_epoch_counter += 1

Loader Num: [40/7999], Train mean loss: 8.7893057, lr: 0.000010
Loader Num: [80/7999], Train mean loss: 7.4978838, lr: 0.000010
Loader Num: [120/7999], Train mean loss: 6.0385437, lr: 0.000010
Loader Num: [160/7999], Train mean loss: 5.0811782, lr: 0.000010
Loader Num: [200/7999], Train mean loss: 4.5222678, lr: 0.000010
Loader Num: [240/7999], Train mean loss: 4.1683979, lr: 0.000010
Loader Num: [280/7999], Train mean loss: 3.8933327, lr: 0.000010
Loader Num: [320/7999], Train mean loss: 3.8929486, lr: 0.000010
Loader Num: [360/7999], Train mean loss: 3.7923913, lr: 0.000010
Loader Num: [400/7999], Train mean loss: 3.4495981, lr: 0.000010
Loader Num: [440/7999], Train mean loss: 3.7016664, lr: 0.000010
Loader Num: [480/7999], Train mean loss: 3.6431587, lr: 0.000010
Loader Num: [520/7999], Train mean loss: 3.4703767, lr: 0.000010
Loader Num: [560/7999], Train mean loss: 3.5424697, lr: 0.000010
Loader Num: [600/7999], Train mean loss: 3.4039013, lr: 0.000010
Loader Num: [640/7999], Tra

In [122]:
# torch.save(parsbert_model.state_dict(), 'models/parsbert_model_ch1.pth')

## ALBERT-Persian

In [28]:
# Load the pre-trained BERT model and tokenizer
config_albert = AutoConfig.from_pretrained("m3hrdadfi/albert-fa-base-v2")
tokenizer_albert = AutoTokenizer.from_pretrained("m3hrdadfi/albert-fa-base-v2")
model_albert = AutoModel.from_pretrained("m3hrdadfi/albert-fa-base-v2")

Some weights of the model checkpoint at m3hrdadfi/albert-fa-base-v2 were not used when initializing AlbertModel: ['predictions.LayerNorm.bias', 'predictions.decoder.bias', 'sop_classifier.classifier.bias', 'sop_classifier.classifier.weight', 'predictions.bias', 'predictions.LayerNorm.weight', 'predictions.dense.weight', 'predictions.dense.bias', 'predictions.decoder.weight']
- This IS expected if you are initializing AlbertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing AlbertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### Preprocessing for ALBERT-Persian

In [16]:
def data_prep_albert(question, context, is_impossible, start_index, stop_index, device):
    encoding = tokenizer_albert.encode_plus(question, context, max_length = 512, truncation=True,
                                            return_tensors='pt', return_offsets_mapping=True)
    offsets = encoding['offset_mapping']
    token_start_index = 0
    token_stop_index = 0
    if is_impossible:
        return [encoding['input_ids'].to(device), encoding['token_type_ids'].to(device), encoding['attention_mask'].to(device), torch.tensor([0], device=device), torch.tensor([0], device=device)]
    for i, (start, end) in enumerate(offsets[0].tolist()):
        if start_index >= start and start_index < end:
            token_start_index = i
        if stop_index > start and stop_index <= end:
            token_stop_index = i
    return [encoding['input_ids'].to(device), encoding['token_type_ids'].to(device), encoding['attention_mask'].to(device), torch.tensor([token_start_index], device=device), torch.tensor([token_stop_index], device=device)]

In [17]:
train_data_raw_pd = pd.read_json('Dataset/Train.json')
train_data_raw_np = train_data_raw_pd.to_numpy()

train_data = []
for row in train_data_raw_np[:,1]:
    for single_context_all_qans in row['paragraphs']:
        context = single_context_all_qans['context']
        for qans in single_context_all_qans['qas']:
            if qans['is_impossible']:
                start_ans_token = 0
                stop_ans_token = 0
                prepared_1data = data_prep_albert(question, context, True, start_ans_idx, stop_ans_idx, device)
                train_data.append(prepared_1data)
                continue
            question = qans['question']
            start_ans_idx = qans['answers'][0]['answer_start']
            stop_ans_idx = start_ans_idx + len(qans['answers'][0]['text'])
            prepared_1data = data_prep_albert(question, context, False, start_ans_idx, stop_ans_idx, device)
            train_data.append(prepared_1data)
            # print(start_ans_idx, stop_ans_idx, question)
        # break
    # break

len(train_data)

63994

In [18]:
validation_data_raw_pd = pd.read_json('Dataset/Validation.json')
validation_data_raw_np = validation_data_raw_pd.to_numpy()

validation_data = []
for row in validation_data_raw_np[:,1]:
    for single_context_all_qans in row['paragraphs']:
        context = single_context_all_qans['context']
        for qans in single_context_all_qans['qas']:
            if qans['is_impossible']:
                start_ans_token = 0
                stop_ans_token = 0
                prepared_1data = data_prep_albert(question, context, True, start_ans_idx, stop_ans_idx, device)
                validation_data.append(prepared_1data)
                continue
            question = qans['question']
            start_ans_idx = qans['answers'][0]['answer_start']
            stop_ans_idx = start_ans_idx + len(qans['answers'][0]['text'])
            prepared_1data = data_prep_albert(question, context, False, start_ans_idx, stop_ans_idx, device)
            validation_data.append(prepared_1data)
            # print(start_ans_idx, stop_ans_idx, question)
        # break
    # break

len(validation_data)

7976

In [19]:
test_data_raw_pd = pd.read_json('Dataset/Test.json')
test_data_raw_np = test_data_raw_pd.to_numpy()

test_data = []
for row in test_data_raw_np[:,1]:
    for single_context_all_qans in row['paragraphs']:
        context = single_context_all_qans['context']
        for qans in single_context_all_qans['qas']:
            if qans['is_impossible']:
                start_ans_token = 0
                stop_ans_token = 0
                prepared_1data = data_prep_albert(question, context, True, start_ans_idx, stop_ans_idx, device)
                test_data.append(prepared_1data)
                continue
            question = qans['question']
            start_ans_idx = qans['answers'][0]['answer_start']
            stop_ans_idx = start_ans_idx + len(qans['answers'][0]['text'])
            prepared_1data = data_prep_albert(question, context, False, start_ans_idx, stop_ans_idx, device)
            test_data.append(prepared_1data)
            # print(start_ans_idx, stop_ans_idx, question)
        # break
    # break

len(test_data)

8002

In [20]:
indices = torch.randperm(len(train_data))
train_data_sh = [train_data[i] for i in indices]

### Trainig for ALBERT-Persian

In [29]:
model_albert

AlbertModel(
  (embeddings): AlbertEmbeddings(
    (word_embeddings): Embedding(80000, 128, padding_idx=0)
    (position_embeddings): Embedding(512, 128)
    (token_type_embeddings): Embedding(2, 128)
    (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0, inplace=False)
  )
  (encoder): AlbertTransformer(
    (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
    (albert_layer_groups): ModuleList(
      (0): AlbertLayerGroup(
        (albert_layers): ModuleList(
          (0): AlbertLayer(
            (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (attention): AlbertAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (attention_dropout): Dropout(p=0, inplace=False)
      

In [30]:
gc.collect()

226

In [31]:
class PosALBERT(nn.Module):
    def __init__(self):
        super(PosALBERT, self).__init__()
        
        self.base_model = model_albert
        self.linear = torch.nn.Sequential(
                      torch.nn.Linear(768, 2),
                        ) 
        
    def forward(self, input_ids, token_type_ids, attention_mask):
        outputs = self.base_model(input_ids, token_type_ids=token_type_ids, attention_mask=attention_mask)
        outputs = self.linear(outputs[0])
        
        return outputs

In [32]:
gc.collect()
torch.cuda.empty_cache()

In [33]:
gc.collect()
torch.cuda.empty_cache()

lr = 0.00001
batch_size = 8
torch.manual_seed(46)
# Initialize the model, the optimizer and criterion
albert_model = PosALBERT()
albert_model = albert_model.to(device)
optimizer = optim.AdamW(albert_model.parameters(), lr=lr, eps=1e-08)
criterion = nn.CrossEntropyLoss()

train_loss = np.zeros((0))
test_loss = np.zeros((0))
total_epoch_counter = 0
total_epoch = 0
loader_size = len(train_data_sh)//batch_size

In [34]:
num_epochs = 1
total_epoch += num_epochs

for epoch in range(num_epochs):
    
    albert_model.train()
    total_loss = 0
    loss_token_start = torch.tensor([0.0],device=device)
    loss_token_stop = torch.tensor([0.0],device=device)
    optimizer.zero_grad()
    for i in range(loader_size):
        torch.cuda.empty_cache()
        for j in range(batch_size):
            input_data = train_data_sh[i*batch_size+j][0]
            token_type_ids = train_data_sh[i*batch_size+j][1]
            attention_mask = train_data_sh[i*batch_size+j][2]
            token_start_true = train_data_sh[i*batch_size+j][3]
            token_stop_true = train_data_sh[i*batch_size+j][4]
            outputs = albert_model(input_data, token_type_ids, attention_mask)
            loss_token_start += criterion(outputs[:,:,0], token_start_true)
            loss_token_stop += criterion(outputs[:,:,1], token_stop_true)
        loss_token_start /= batch_size
        loss_token_stop /= batch_size
        loss = loss_token_start + loss_token_stop
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        loss_token_start = torch.tensor([0.0],device=device)
        loss_token_stop = torch.tensor([0.0],device=device)
        total_loss += loss
        if (i+1)*batch_size > 32000:
            break
    
        if (i+1) % 40 == 0:
            total_loss /= 40
            print('Loader Num: [{}/{}], Train mean loss: {:.7f}, lr: {:.6f}'
                   .format(i+1, loader_size, total_loss.detach().item(), lr))
            total_loss = 0
    total_epoch_counter += 1

Loader Num: [40/7999], Train mean loss: 9.6707401, lr: 0.000010
Loader Num: [80/7999], Train mean loss: 8.1403608, lr: 0.000010
Loader Num: [120/7999], Train mean loss: 6.5024638, lr: 0.000010
Loader Num: [160/7999], Train mean loss: 5.1578059, lr: 0.000010
Loader Num: [200/7999], Train mean loss: 4.2500529, lr: 0.000010
Loader Num: [240/7999], Train mean loss: 4.1606202, lr: 0.000010
Loader Num: [280/7999], Train mean loss: 4.0323052, lr: 0.000010
Loader Num: [320/7999], Train mean loss: 3.9594471, lr: 0.000010
Loader Num: [360/7999], Train mean loss: 3.6013238, lr: 0.000010
Loader Num: [400/7999], Train mean loss: 3.5468240, lr: 0.000010
Loader Num: [440/7999], Train mean loss: 3.3022430, lr: 0.000010
Loader Num: [480/7999], Train mean loss: 3.6419551, lr: 0.000010
Loader Num: [520/7999], Train mean loss: 3.0675728, lr: 0.000010
Loader Num: [560/7999], Train mean loss: 3.3380387, lr: 0.000010
Loader Num: [600/7999], Train mean loss: 3.1266613, lr: 0.000010
Loader Num: [640/7999], Tra

In [35]:
# torch.save(albert_model.state_dict(), 'models/albert_model_ch1.pth')

In [ ]:
albert_val_em, parsbert_val_f1 = evaluator(albert_model, validation_data)

Exact match accuracy: 44.06%
F1 score: 93.80%


In [ ]:
albert_test_em, parsbert_test_f1 = evaluator(albert_model, test_data)

Exact match accuracy: 44.14%
F1 score: 92.61%


# Evaluation and Postprocessing

In [179]:
def ques2ans(model, tokenizer, question, context, device='cuda:0'):
    new_data = tokenizer.encode_plus(question, context, max_length = 512, truncation=True,
                                       return_tensors='pt', return_offsets_mapping=True)
    model = model.to(device)
    model.eval()
    output = model(new_data['input_ids'].to(device), new_data['token_type_ids'].to(device), new_data['attention_mask'].to(device))
    start_logits, end_logits = output[:,:,0], output[:,:,1]
    # Extract the answer span
    start_index = torch.argmax(start_logits, dim=1).item()
    end_index = torch.argmax(end_logits, dim=1).item()
    if start_index == 0 and end_index == 0:
        return '[No Answer]'
    answer = tokenizer.decode(new_data['input_ids'][0][start_index:end_index+1])
    return answer

In [180]:
quest = 'اسم شاعر ایرانی چیست؟'
paragraph = 'یک شاعر ایرانی معروف در طوس دفن شده است. نام آن شاعر فردوسی است و اثر او شاهنامه است'
ques2ans(albert_model, tokenizer_albert, quest, paragraph)

'فردوسی'

In [181]:
quest = 'اسم شاعر ایرانی چیست؟'
paragraph = 'یک شاعر ایرانی معروف در طوس دفن شده است و اثر او شاهنامه است'
ques2ans(albert_model, tokenizer_albert, quest, paragraph)

'[No Answer]'

## ParsBERT evaluation

In [36]:
def evaluator(model, test_data):
    exact_match = 0
    f1_score = 0
    model.eval()
    # Iterate over the test examples
    for single_input in test_data:
        # Compute the start and end logits
        outputs = model(single_input[0], single_input[1], single_input[2])
        start_logits, end_logits = outputs[:,:,0], outputs[:,:,1]

        # Get the predicted span by finding the indices with the highest logits
        start_index = torch.argmax(start_logits, dim=1).item()
        end_index = torch.argmax(end_logits, dim=1).item()
        # predicted_span = tokenizer.decode(single_input[0][0][start_index:end_index+1])
        predicted_span = single_input[0][0][start_index:end_index+1]

        # Get the ground truth span
        ground_truth_span = single_input[0][0][single_input[3]:single_input[4]+1]
        # print(predicted_span, ground_truth_span)
        # break

        # Compute the exact match and F1 scores
        if predicted_span.shape == ground_truth_span.shape:
            if (predicted_span == ground_truth_span).sum() == len(predicted_span):
                exact_match += 1

        predicted_tokens = predicted_span
        ground_truth_tokens = ground_truth_span
        common = Counter(predicted_tokens) and Counter(ground_truth_tokens)
        num_same = sum(common.values())

        if num_same == 0:
            f1 = 0
        else:
            precision = 1.0 * num_same / len(predicted_tokens)
            recall = 1.0 * num_same / len(ground_truth_tokens)
            f1 = (2 * precision * recall) / (precision + recall)

        f1_score += f1
    # Compute the average scores over all test examples
    exact_match /= len(test_data)
    f1_score /= len(test_data)

    print("Exact match accuracy: {:.2f}%".format(100 * exact_match))
    print("F1 score: {:.2f}%".format(100 * f1_score))
    return exact_match, f1_score

In [108]:
parsbert_val_em, parsbert_val_f1 = evaluator(parsbert_model, validation_data)

Exact match accuracy: 44.31%
F1 score: 92.39%


In [107]:
parsbert_test_em, parsbert_test_f1 = evaluator(parsbert_model, test_data)

Exact match accuracy: 44.50%
F1 score: 92.19%


## ALBERT-Persian evaluation

In [ ]:
albert_val_em, albert_val_f1 = evaluator(albert_model, validation_data)

Exact match accuracy: 44.06%
F1 score: 93.80%


In [ ]:
albert_test_em, albert_test_f1 = evaluator(albert_model, test_data)

Exact match accuracy: 44.14%
F1 score: 92.61%
